# Expectation Maximization
*ovvero la rigorosa formalizzazione matematica del cerchiobottismo.*

<img src="images/barrels.jpg" width="600"/>


Expectation Maximization (EM) is an iterative algorithm for maximum likelihood estimation when some variables are latent. Because marginalizing over latent variables makes the log-likelihood intractable, EM instead maximizes a tractable lower bound called the Evidence Lower Bound (ELBO):

$$\log p(x|\theta) = \mathcal{L}(q, \theta) + \mathrm{KL}[q(z) \| p(z|x,\theta)]$$

The algorithm initializes $\theta_{t=0}$ and then it alternates two steps until convergence:
- **E-step**: set $q_t = p(z | x, \theta_t)$. This makes KL = 0, so ELBO = log-likelihood.
- **M-step**: maximize $ \theta_t =\arg \max _\theta \mathcal{L}(q_t, \theta) =  \arg \max _\theta E_{z \sim q_t}[\log p(x, z | \theta)]$. This maximizes the ELBO, i.e the log-likelihood.

The algorithm converges to a local optimum.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

rng = np.random.default_rng(42)

---
## Two Biased Coins

### Setup

We have two biased coins $A$ and $B$ with unknown head probabilities $\theta_A$ and $\theta_B$. In each of $5$ independent trials a coin is chosen at random (with equal prior probability) and flipped $n=10$ times. At each trial $i$ we observe only the head counts $x_i$, not which coin was used. The coin identity is the latent variable $z_i \in \{A, B\}$.

Model
- $z_i \sim Bernoulli(0.5)$ (unobserved)
- $x_i | z_i = A \sim Binomial(10, \theta_A)$
- $x_i | z_i = B \sim Binomial(10, \theta_B)$

Goal: recover $\theta_A$ and $\theta_B$ from the observed head counts $x_1, ..., x_5$.

### E-step
For each trial $i$, compute responsibility:

$$
r_i^A = p(z_i=A |x_i) = \frac{p(z_i=A) \binom{n}{x_i} \theta_A^{x_i} (1 - \theta_A)^{(n - x_i)}}{p(x_i)}  \propto  \theta_A^{x_i} (1 - \theta_A)^{(n - x_i)}
$$

$$
r_i^B  \propto  \theta_B^{x_i} (1 - \theta_B)^{(n - x_i)}
$$

and normalize.

### M-step
Update the bias estimates as responsibility-weighted MLE:
$$
\theta_A = \frac{\sum_i r_i^A x_i}{n\sum_i r_i^A} = \frac{\text{Total heads from A}}{\text{Total flips from A}}
$$
$$
\theta_B = \frac{\sum_i r_i^B x_i}{n\sum_i r_i^B}
$$

### Observations

$$\{H,T,H,T,H,H,T,T,H,T\} \rightarrow x_1 = 5$$
$$\{H,T,H,H,H,H,H,H,H,H\}\rightarrow x_2 = 9$$
$$\{H,T,H,H,H,H,H,T,H,H\}\rightarrow x_3 = 8$$
$$\{H,T,H,T,H,T,T,T,H,T\}\rightarrow x_4 = 4$$
$$\{H,T,H,T,H,H,H,T,H,H\}\rightarrow x_5 = 7$$

###

In [ ]:
# Observed head counts (from Dempster et al. 1977 classic example)
heads = np.array([5, 9, 8, 4, 7], dtype=float)
n_flips = 10

# EM for two biased coins
def em_coins(heads, n_flips, theta_A_init=0.1, theta_B_init=0.2, n_iter=20):
    theta_A = theta_A_init
    theta_B = theta_B_init
    history = [(theta_A, theta_B)]
    log_likelihoods = []

    for _ in range(n_iter):
        # E-step: compute responsibilities and normalize them
        pass


        # Log-likelihood of observed data (marginalizing over z)


        # M-step: update theta estimates and append to history
        


    return np.array(history), np.array(log_likelihoods)


In [ ]:
history, log_likelihoods = em_coins(heads, n_flips)

print(f"Final estimates:  theta_A = {history[-1][0]:.4f}   theta_B = {history[-1][1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Parameter convergence
ax = axes[0]
ax.plot(history[:, 0], marker='o', label='theta_A')
ax.plot(history[:, 1], marker='s', label='theta_B')
ax.set_xlabel('Iteration')
ax.set_ylabel('Bias estimate')
ax.set_title('Coin bias estimates vs. iteration')
ax.legend()
ax.grid(True, alpha=0.3)

# Log-likelihood (should be non-decreasing)
ax = axes[1]
ax.plot(log_likelihoods, marker='o', color='C2')
ax.set_xlabel('Iteration')
ax.set_ylabel('Log-likelihood')
ax.set_title('Log-likelihood vs. iteration')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
Mixture of Gaussians

### Setup

A Gaussian Mixture Model (GMM) assumes that each observation $x_n$ is drawn from one of $K$ Gaussian components. The component assignment $z_n \in \{1,...,K\}$ is latent. The full model is:
$$
p(x, z | \theta) = \prod_j  \pi_j^{z_j}   \mathcal{N}(x | \mu_j, \sigma_j^2)^{z_j}
$$
where $z$ is one hot encoding vector and $\pi_j = p(z_j = 1)$, i.e how common component $j$ is in the population?.
Since $z$ is not observed, the marginal
$$
p(x | \theta) = \sum_j  \pi_j  \mathcal{N}(x | \mu_j, \sigma_j^2)
$$
is a sum inside a log, making direct MLE intractable. EM handles this naturally.

### E-step: compute responsibilities
$$
\gamma(z_{n,j}) = \frac{\pi_j \mathcal{N}(x_n | \mu_j, \sigma_j^2)}{\sum_i \pi_i \mathcal{N}(x_n | \mu_i, \sigma_i^2)}
$$
Given that I saw this specific $x_n$​, how likely is it that component $j$ generated it?
### M-step: closed-form updates of parameters
$$
N_j = \sum_n \gamma(z_{n,j})
$$
$$
\mu_{j_\text{new}}  = \frac{1}{N_j}  \sum_n \gamma(z_{n,j})  x_n
$$
$$
\sigma^2_{j_\text{new}} = \frac{1}{N_j}  * \sum_n \gamma(z_nj) (x_n - \mu_{j_\text{new}} )^2
$$
$$
\pi_{j_\text{new}}   = \frac{N_j}{N} 
$$


### ELBO
$$
ELBO = \sum_n \sum_j \gamma(z_{n,j}) * [\log \pi_j + \log N(x_n | \mu_j, \sigma_j^2) - \log \gamma(z_{n,j})]
$$

In [ ]:
# Ground-truth parameters (used only for data generation)
K = 3
true_mu    = np.array([-8.0, 0.0, 6.0])
true_sigma = np.array([0.8, 1.2, 0.6])
true_pi    = np.array([0.3, 0.4, 0.3])
N = 400

# Generate data
z_true = rng.choice(K, size=N, p=true_pi)
X = rng.normal(true_mu[z_true], true_sigma[z_true])

print(f"Generated {N} samples from a {K}-component GMM")
print(f"True mu:    {true_mu}")
print(f"True sigma: {true_sigma}")
print(f"True pi:    {true_pi}")

In [ ]:
def gaussian_pdf(x, mu, sigma):
    return norm.pdf(x, loc=mu, scale=sigma)


def em_gmm(X, K, n_iter=50, seed=7):
    rng_local = np.random.default_rng(seed)
    N = len(X)

    # Random initialization
    pass

    elbo_history = []
    snapshots = []  # store (mu, sigma, pi, gamma) at selected iterations

    for it in range(n_iter):
        # E-step: responsibilities
        # gamma_mat shape: (N, K)
        pass


        # ELBO


        # Append some snapshot of the parameters for later visualization
        if it in (0, 1, 2, 4, 9, n_iter - 1):
            snapshots.append((it, mu.copy(), sigma.copy(), pi.copy(), gamma_mat.copy()))
            

        # M-step: parameter updates




    return mu, sigma, pi, np.array(elbo_history), snapshots


In [ ]:
mu_fit, sigma_fit, pi_fit, elbo_history, snapshots = em_gmm(X, K, n_iter=50)

print("\nRecovered parameters:")
print(f"  mu:    {np.sort(mu_fit).round(3)}")
print(f"  sigma: {sigma_fit[np.argsort(mu_fit)].round(3)}")
print(f"  pi:    {pi_fit[np.argsort(mu_fit)].round(3)}")

In [ ]:
# Multi-panel plot: GMM fit at selected iterations
x_grid = np.linspace(X.min() - 1, X.max() + 1, 400)
colors = ['C0', 'C1', 'C2']

n_snap = len(snapshots)
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

for ax, (it, mu_s, sigma_s, pi_s, gamma_s) in zip(axes, snapshots):
    ax.hist(X, bins=40, density=True, color='lightgray', edgecolor='white', label='data')

    # Plot individual components and mixture
    mixture = np.zeros_like(x_grid)
    for j in range(K):
        comp = pi_s[j] * gaussian_pdf(x_grid, mu_s[j], sigma_s[j])
        ax.plot(x_grid, comp, '--', color=colors[j], alpha=1.0,
                label=f'comp {j+1}')
        mixture += comp
    ax.plot(x_grid, mixture, 'k-', linewidth=1, label='mixture', alpha=0.5)

    ax.set_title(f'Iteration {it}')
    ax.set_xlabel('x')
    ax.set_ylabel('density')
    ax.grid(True, alpha=0.3)
    if it == 0:
        ax.legend(fontsize=7)

plt.suptitle('GMM fitted by EM — snapshots over iterations', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ELBO vs iteration
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(elbo_history, marker='o', markersize=4, color='C3')
ax.set_xlabel('Iteration')
ax.set_ylabel('ELBO')
ax.set_title('ELBO vs. iteration')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Summary: ground truth vs recovered
order = np.argsort(mu_fit)
print("Parameter recovery (sorted by mu):")
print(f"{'':10s}  {'mu (true)':>12s}  {'mu (fit)':>10s}  {'sigma (true)':>14s}  {'sigma (fit)':>12s}  {'pi (true)':>10s}  {'pi (fit)':>10s}")
for rank, j in enumerate(order):
    t = rank  # assumes sorted true params
    print(f"Component {rank+1}  {true_mu[rank]:>12.3f}  {mu_fit[j]:>10.3f}  "
          f"{true_sigma[rank]:>14.3f}  {sigma_fit[j]:>12.3f}  "
          f"{true_pi[rank]:>10.3f}  {pi_fit[j]:>10.3f}")